# 8.01 VLA + RL 全景：为什么、做什么、怎么选

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/08_vla_rl/8.01_vla_rl_overview.ipynb)


## 为什么 SFT 之后还需要 RL？

模仿学习（SFT）的三个天花板：

1. **示教质量的天花板**：模型最多和示教者一样好。人类遥操作慢、抖、偶尔犯错，插线时常常"差一点没插到底"。
2. **分布偏移**（第 3 章）：部署时模型会进入示教中没有的状态，SFT 不知道怎么从中恢复。
3. **精度与成功率的"最后一公里"**：VLA 能做到 70%~80% 很快，从 80% 到 99% 极其困难，而落地需要的恰恰是后者。

RL 用**奖励**（插好了 = 1）直接优化我们关心的指标，可以突破示教的上限。

---

## 主流方法地图

| 方法 | 改的是什么 | on / off-policy | 需要什么 | 代表工作 |
|---|---|---|---|---|
| **PPO / GRPO 直接微调 VLA** | 整个 VLA（或 LoRA） | on | 大量仿真交互；流匹配 VLA 需要把 ODE 改造成可算 log π 的随机过程 | RLinf（π0 + flow-noise/flow-SDE）、SimpleVLA-RL、VLA-RL |
| **残差 RL** | 在 VLA 输出上加一个小的修正网络 | off | 少量真机交互 | ResiP、Policy Decorator |
| **RLT（RL Token）** | 冻结 VLA；小 actor-critic 以 VLA 的"RL token"和参考动作为输入 | off | 真机交互 + 人工干预 | RLT（PI 2025）、RLinf 的 RLT 实现 |
| **DSRL（噪声空间 RL）** | 冻结 VLA；学一个"选初始噪声"的策略 | off | 少量交互 | DSRL（Wagenmaker et al. 2025） |
| **RECAP（优势条件化）** | 整个 VLA，用"优势好/坏"作为条件做 SFT | 离线迭代 | 自主采集的数据 + 人工纠正 + 价值函数 | π\*0.6（PI 2025） |
| **HIL-SERL** | 从零训练小策略（不用 VLA） | off | 真机交互 + 人工干预 | HIL-SERL（Luo et al. 2024） |

### 一个统一的视角

几乎所有方法都可以写成"**带权重的行为克隆 + 保持在 VLA 附近**"：

| 方法 | "往哪里学" | "不要离 VLA 太远"的机制 |
|---|---|---|
| PPO | $\nabla\log\pi\cdot\hat A$（优势加权） | 裁剪比值、KL 惩罚 |
| RLT | $-Q(s,a)$（沿 Q 上升） | BC 正则 $\beta\|a-a_{ref}\|^2$、固定小 σ、残差零初始化 |
| DSRL | 噪声空间里的 $-Q$ | 动作永远来自 VLA 的生成过程 |
| RECAP | 只模仿"优势为正"的数据 | 本身就是 SFT |

---

## 流匹配 VLA 怎么做 PPO？——log π 的问题

PPO 需要 $\log\pi(\mathbf a|s)$（2.11 课）。但 π0 的动作是**从噪声积分 ODE 得到的**：给定初始噪声，过程完全确定，没有可以直接计算的概率密度。

RLinf 等框架的做法：把确定性的 Euler 步改成**随机**的，每一步都是一个高斯转移：

$$
\mathbf x_{t+\Delta t}\sim\mathcal N\big(\mathbf x_t+\Delta t\,\mathbf v_\theta(\mathbf x_t,t),\ \sigma_t^2 I\big)
$$

于是整条去噪链的对数概率就是**各步高斯对数概率之和**：

$$
\log\pi(\mathbf x_{0:1}\mid s)=\sum_{k}\log\mathcal N\big(\mathbf x_{k+1};\ \mathbf x_k+\Delta t\,\mathbf v_\theta(\mathbf x_k,t_k),\ \sigma_k^2I\big)
$$

（flow-SDE 由 ODE 推出与之边缘分布一致的 SDE 来确定 $\sigma_t$ 和漂移修正项；flow-noise 则用一个网络学习每步的噪声大小。这里只演示最简单的形式。）

### 练习：随机 Euler 去噪链的对数概率

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import math
import torch
from torch.distributions import Normal

def stochastic_euler_logprob(velocity_fn, x_traj, sigma, num_steps):
    """
    velocity_fn(x, t) -> 速度；x_traj: (K+1, B, D) 一条随机去噪链 x_1 -> ... -> x_0（openpi 约定 t 从 1 到 0）
    sigma: 每步的噪声标准差（标量）。返回每个样本的 log π，形状 (B,)
    """
    dt = -1.0 / num_steps
    logp = torch.zeros(x_traj.shape[1])
    # TODO: 对 k = 0..K-1：t = 1 + k*dt；mean = x_traj[k] + dt * velocity_fn(x_traj[k], t)
    #       logp += Normal(mean, sigma).log_prob(x_traj[k+1]).sum(-1)
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
import math
import torch
from torch.distributions import Normal

def stochastic_euler_logprob(velocity_fn, x_traj, sigma, num_steps):
    """
    velocity_fn(x, t) -> 速度；x_traj: (K+1, B, D) 一条随机去噪链 x_1 -> ... -> x_0（openpi 约定 t 从 1 到 0）
    sigma: 每步的噪声标准差（标量）。返回每个样本的 log π，形状 (B,)
    """
    dt = -1.0 / num_steps
    logp = torch.zeros(x_traj.shape[1])
    for k in range(num_steps):
        t = 1.0 + k * dt
        mean = x_traj[k] + dt * velocity_fn(x_traj[k], t)
        logp = logp + Normal(mean, sigma).log_prob(x_traj[k + 1]).sum(-1)
    return logp

In [ ]:
# ✅ 自测：用一个线性速度场 v(x,t) = w*x 生成随机链，检查 log π 对 w 可导（PPO 需要它）
torch.manual_seed(0)
w = torch.tensor(0.5, requires_grad=True)
vel = lambda x, t: w * x
K, B, D, sigma = 10, 4, 2, 0.1
with torch.no_grad():
    xs = [torch.randn(B, D)]
    for k in range(K):
        xs.append(xs[-1] + (-1.0 / K) * vel(xs[-1], 1 - k / K) + sigma * torch.randn(B, D))
traj = torch.stack(xs)
lp = stochastic_euler_logprob(vel, traj, sigma, K)
lp.sum().backward()
assert lp.shape == (B,) and w.grad is not None and torch.isfinite(w.grad)
print(f"通过 ✅ log π = {lp.detach().numpy().round(1)}，d(log π)/dw = {w.grad.item():.2f}")

> 有了 $\log\pi$，2.11 课的 PPO 就可以原样使用。RLinf 在 ManiSkill / LIBERO 上用这种方式把 π0 的成功率从 SFT 的水平进一步提升。但 on-policy 方法需要**大量交互**（数万到数百万步），在真机上很难承受，所以真机上更多使用下面的 off-policy 方法。

---

## 怎么选？一个决策流程

```text
你的任务能在仿真里跑吗？
├─ 能，而且仿真足够真实 ─────────────> PPO / GRPO 直接微调 VLA（RLinf），再 sim-to-real
└─ 只能在真机上
   ├─ VLA 已经"大致会做"，只差精度/成功率（比如 60%~80%）
   │    ├─ 问题主要在"最后一段"的精细动作 ──> RLT（冻结 VLA，小网络在参考动作附近修正）  ← 你的线头对插
   │    └─ 想要整体变快、更鲁棒 ─────────────> RECAP（自主运行 + 人工纠正 + 优势条件化，迭代多轮）
   └─ VLA 基本不会做（<20%）─────────────────> 先补示教数据 / DAgger（第 3 章）；或 HIL-SERL 从零学一个小策略
```

**关于"不微调 VLA 本体"**：RLT、DSRL、残差 RL 都**冻结 VLA**，只训练一个很小的网络。好处：

- 不会破坏 VLA 的泛化能力和其它技能；
- 训练很快（小网络、off-policy），适合真机；
- 回滚容易：去掉小网络就回到原来的 VLA。

RECAP 和 PPO 则会**修改 VLA 本身**。如果你有"保留 base 不动"的约束，可以微调一份**副本**，或者只训练 LoRA 适配器。

---

## 小结

- RL 突破 SFT 的三个天花板：示教质量、分布偏移、最后一公里的成功率。
- 统一视角：带权重的模仿 + 不要离 VLA 太远。
- 流匹配 VLA 做 PPO 的关键：把去噪过程改成随机的，log π = 各步高斯之和。
- 真机首选 off-policy、冻结 VLA 的方法（RLT / DSRL / 残差 RL）；想整体提升则用 RECAP 迭代。

下一课：从零实现 RLT。